## 01 - Hyperparameter Study (Deep Notched)

This notebook defines three W&B sweep studies for deep-notched CM runs:
1. isotropic + forward
2. isotropic + inverse
3. orthotropic + inverse

Sweep logs are stored under `results/deep_notched/sweep/<study_name>/`.

In [ ]:
from pathlib import Path
import os
import json
import yaml
import wandb

from phd.config import load_config

PROJECT = "deep-notched"
ENTITY = None  # e.g. "your-wandb-entity"

chapter_dir = Path.cwd()
if chapter_dir.name != "02_deep_notched":
    chapter_dir = Path("chapters/IV_MaterialCharacterization/02_deep_notched").resolve()

agent_script = chapter_dir / "deep_notched_wandb_agent.py"
sweep_root = Path("results/deep_notched/sweep")
sweep_root.mkdir(parents=True, exist_ok=True)

print(f"Agent script: {agent_script}")
print(f"Sweep root:   {sweep_root.resolve()}")

### Sweep space

Shared hyperparameters (as requested):
- `model.architecture.n_hidden`: [3, 4]
- `model.architecture.mlp_type`: ["mlp", "modified-mlp"]
- `model.fourier_features.n_features`: [64, 128, 256]
- `training.lr_decay`: [["warmup cosine", 1e-4, 5e-3, 1000, 30000, 1e-4], None]
- `training.num_domain`: [200, 300]
- `seed`: [0, 1, 2]

In [ ]:
shared_parameters = {
    "model.architecture.n_hidden": {"values": [3, 4]},
    "model.architecture.mlp_type": {"values": ["mlp", "modified-mlp"]},
    "model.fourier_features.n_features": {"values": [64, 128, 256]},
    "training.lr_decay": {"values": [
        ["warmup cosine", 1e-4, 5e-3, 1000, 30000, 1e-4],
        None,
    ]},
    "training.num_domain": {"values": [200, 300]},
    "seed": {"values": [0, 1, 2]},
}

common_fixed = {
    "problem.name": {"value": "deep_notched"},
    "model.net_type": {"value": "SPINN"},
    "model.fourier_features.enabled": {"value": True},
    "training.self_attention.enabled": {"value": False},
    "training.n_iter": {"value": 30000},
}

studies = {
    "iso_forward": {
        "task.type": {"value": "forward"},
        "problem.material.law": {"value": "isotropic"},
        "task.measurements.enabled": {"value": False},
    },
    "iso_inverse": {
        "task.type": {"value": "inverse"},
        "problem.material.law": {"value": "isotropic"},
        "task.measurements.enabled": {"value": True},
    },
    "ortho_inverse": {
        "task.type": {"value": "inverse"},
        "problem.material.law": {"value": "orthotropic"},
        "task.measurements.enabled": {"value": True},
    },
}

def build_sweep_config(study_name, metric_name="l2_relative_error"):
    params = {}
    params.update(common_fixed)
    params.update(studies[study_name])
    params.update(shared_parameters)
    return {
        "name": f"deep_notched_{study_name}",
        "method": "grid",
        "metric": {"name": metric_name, "goal": "minimize"},
        "parameters": params,
    }

def count_runs(sweep_cfg):
    n = 1
    for p in sweep_cfg["parameters"].values():
        n *= len(p["values"]) if "values" in p else 1
    return n

for study in studies:
    cfg = build_sweep_config(study)
    print(f"{study}: {count_runs(cfg)} runs")

In [ ]:
CREATE_SWEEPS = False

sweep_ids = {}
for study in studies:
    sweep_cfg = build_sweep_config(study)
    study_dir = sweep_root / study
    study_dir.mkdir(parents=True, exist_ok=True)

    # Store config locally for reproducibility
    with open(study_dir / "sweep_config.yaml", "w") as f:
        yaml.safe_dump(sweep_cfg, f, sort_keys=False)

    if CREATE_SWEEPS:
        sid = wandb.sweep(sweep=sweep_cfg, project=PROJECT, entity=ENTITY)
        sweep_ids[study] = sid
        print(f"Created {study}: {sid}")
    else:
        # Fill these manually or keep from a previous run
        sweep_ids[study] = None

    with open(study_dir / "sweep_info.json", "w") as f:
        json.dump({
            "study": study,
            "project": PROJECT,
            "entity": ENTITY,
            "sweep_id": sweep_ids[study],
        }, f, indent=2)

sweep_ids

In [ ]:
# Fill these after creating sweeps (or from existing sweeps)
sweep_ids = {
    "iso_forward": sweep_ids.get("iso_forward") or "<ISO_FORWARD_SWEEP_ID>",
    "iso_inverse": sweep_ids.get("iso_inverse") or "<ISO_INVERSE_SWEEP_ID>",
    "ortho_inverse": sweep_ids.get("ortho_inverse") or "<ORTHO_INVERSE_SWEEP_ID>",
}

print("Run each command in a terminal:")
for study, sid in sweep_ids.items():
    cmd = (
        f"python {agent_script} --study {study} --sweep-id {sid} "
        f"--project {PROJECT}"
    )
    if ENTITY:
        cmd += f" --entity {ENTITY}"
    print(f"\n[{study}]\n{cmd}")

### Notes

- The terminal agent logs **metrics only** (no full history).
- For inverse studies, identified material variables are logged as `identified_<name>`.
- All W&B run files are written to `results/deep_notched/sweep/<study>/`.
- If you also want `orthotropic + forward`, duplicate one study block and set:
  - `task.type = forward`
  - `problem.material.law = orthotropic`
  - `task.measurements.enabled = False`